# Enclave Startup Time Analysis

Loads `logs.csv` produced by the benchmark, extracts per-request startup latencies (`diff_ms` from "listener hit, reporting diff" entries), and visualises distribution, trends, and throughput.

**Key metric:** `diff_ms` — wall-clock milliseconds from task publish until the enclave callback fires.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

plt.rcParams.update({'figure.dpi': 120, 'axes.grid': True, 'grid.alpha': 0.3})

DATA_PATH = Path('..') / 'logs.csv'
assert DATA_PATH.exists(), f'logs.csv not found at {DATA_PATH.absolute()}'

df = pd.read_csv(DATA_PATH)
df['timestamp'] = pd.to_datetime(df['timestamp'])

def parse_fields(x):
    if pd.isna(x) or x == '':
        return {}
    try:
        return json.loads(x)
    except Exception:
        return {}

df['fields_parsed'] = df['fields'].apply(parse_fields)

print(f'Total log rows : {len(df):,}')
print(f'Time range     : {df["timestamp"].min()}  →  {df["timestamp"].max()}')
print(f'Duration       : {df["timestamp"].max() - df["timestamp"].min()}')
print()
print('Message counts:')
print(df['message'].value_counts().to_string())

In [ ]:
# --- Extract startup latencies ---
# "listener hit, reporting diff" rows carry the authoritative diff_ms field.
hits = df[df['fields_parsed'].apply(lambda d: 'diff_ms' in d)].copy()
hits['diff_ms'] = hits['fields_parsed'].apply(lambda d: float(d['diff_ms']))
hits['request'] = hits['fields_parsed'].apply(lambda d: d.get('request', ''))

# One canonical measurement per request: keep the *first* callback hit.
hits = hits.sort_values('timestamp').drop_duplicates('request', keep='first').reset_index(drop=True)

print(f'Unique completed requests : {len(hits):,}')
print()

# --- Summary statistics ---
stats = hits['diff_ms'].describe(percentiles=[.50, .75, .90, .95, .99])
stats.index = ['count', 'mean', 'std', 'min', 'p50', 'p75', 'p90', 'p95', 'p99', 'max']
stats_df = stats.rename('ms').to_frame()
stats_df['s'] = (stats_df['ms'] / 1000).round(3)
print(stats_df.to_string(float_format=lambda x: f'{x:,.1f}'))

In [ ]:
# --- Distribution: Histogram + KDE ---
diffs = hits['diff_ms'].values

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Histogram
ax = axes[0]
ax.hist(diffs, bins=60, color='#4c9be8', edgecolor='white', linewidth=0.4)
for p, label, color in [
    (np.median(diffs), f'p50 {np.median(diffs)/1000:.1f}s', '#e06c1e'),
    (np.percentile(diffs, 95), f'p95 {np.percentile(diffs, 95)/1000:.1f}s', '#c0392b'),
]:
    ax.axvline(p, color=color, linestyle='--', linewidth=1.4, label=label)
ax.set_xlabel('Startup time (ms)')
ax.set_ylabel('Count')
ax.set_title('Startup time distribution')
ax.legend(fontsize=9)

# Empirical CDF
ax = axes[1]
vals = np.sort(diffs)
cdf  = np.arange(1, len(vals) + 1) / len(vals)
ax.plot(vals, cdf * 100, color='#4c9be8', linewidth=1.5)
for pct in [50, 90, 95, 99]:
    v = np.percentile(diffs, pct)
    ax.axvline(v, linestyle=':', linewidth=1, alpha=0.7, label=f'p{pct} {v/1000:.1f}s')
ax.set_xlabel('Startup time (ms)')
ax.set_ylabel('Percentile (%)')
ax.set_title('Empirical CDF')
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# --- Startup time over time (rolling median + p95 band) ---
hits_sorted = hits.sort_values('timestamp').copy()
hits_sorted.set_index('timestamp', inplace=True)

# Rolling window of 100 samples
window = 100
roll = hits_sorted['diff_ms'].rolling(window)
rolling_med = roll.median()
rolling_p95 = roll.quantile(0.95)
rolling_min = roll.min()

fig, ax = plt.subplots(figsize=(13, 4))
ax.fill_between(rolling_min.index, rolling_min, rolling_p95,
                alpha=0.15, color='#4c9be8', label='min–p95 band')
ax.plot(rolling_med.index, rolling_med / 1000,
        color='#4c9be8', linewidth=1.4, label=f'rolling median (w={window})')
ax.plot(rolling_p95.index, rolling_p95 / 1000,
        color='#c0392b', linewidth=1, linestyle='--', label=f'rolling p95 (w={window})')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
ax.xaxis.set_major_locator(mdates.AutoDateLocator())
fig.autofmt_xdate()
ax.set_xlabel('Time')
ax.set_ylabel('Startup time (s)')
ax.set_title('Startup time over the benchmark run')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# --- Throughput: completed tasks per minute ---
hits_by_min = (
    hits.set_index('timestamp')['diff_ms']
    .resample('1min')
    .count()
    .rename('tasks_per_min')
)

fig, ax = plt.subplots(figsize=(13, 3))
ax.bar(hits_by_min.index, hits_by_min.values,
       width=pd.Timedelta(seconds=50), color='#5ab26e', alpha=0.85)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
ax.xaxis.set_major_locator(mdates.AutoDateLocator())
fig.autofmt_xdate()
ax.set_xlabel('Time')
ax.set_ylabel('Completed tasks')
ax.set_title('Throughput — completed tasks per minute')
plt.tight_layout()
plt.show()

print(f'Peak throughput  : {hits_by_min.max():.0f} tasks/min')
print(f'Avg throughput   : {hits_by_min.mean():.1f} tasks/min')

In [ ]:
# --- Boxplot + Violin side-by-side ---
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Boxplot
ax = axes[0]
bp = ax.boxplot(diffs / 1000, vert=True, patch_artist=True,
                boxprops=dict(facecolor='#a6cee3', alpha=0.8),
                medianprops=dict(color='#e06c1e', linewidth=2),
                flierprops=dict(marker='.', markersize=2, alpha=0.3))
ax.set_ylabel('Startup time (s)')
ax.set_xticks([1]); ax.set_xticklabels(['startup'])
ax.set_title('Box plot')

# Violin
ax = axes[1]
parts = ax.violinplot(diffs / 1000, showmedians=True, showextrema=False)
for pc in parts['bodies']:
    pc.set_facecolor('#a6cee3'); pc.set_alpha(0.8)
parts['cmedians'].set_color('#e06c1e'); parts['cmedians'].set_linewidth(2)
ax.set_ylabel('Startup time (s)')
ax.set_xticks([1]); ax.set_xticklabels(['startup'])
ax.set_title('Violin plot')

plt.suptitle('Startup time spread', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# --- Outlier / SLA breach analysis ---
# Define SLA thresholds (adjust as needed)
sla_thresholds_s = [1, 2, 5, 10, 30]

print(f'{"SLA (s)":<10} {"Breaches":>10} {"Breach rate":>13}')
print('-' * 35)
for t in sla_thresholds_s:
    breaches = (hits['diff_ms'] > t * 1000).sum()
    rate = breaches / len(hits) * 100
    print(f'{t:<10} {breaches:>10,} {rate:>12.2f}%')

# Scatter of outliers (top 1%)
threshold_99 = np.percentile(diffs, 99)
outliers = hits[hits['diff_ms'] > threshold_99]

fig, ax = plt.subplots(figsize=(13, 3))
ax.scatter(hits['timestamp'], hits['diff_ms'] / 1000,
           s=1, alpha=0.2, color='#4c9be8', label='all requests')
ax.scatter(outliers['timestamp'], outliers['diff_ms'] / 1000,
           s=6, alpha=0.7, color='#c0392b', label=f'top 1% outliers (>{threshold_99/1000:.1f}s)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
ax.xaxis.set_major_locator(mdates.AutoDateLocator())
fig.autofmt_xdate()
ax.set_xlabel('Time')
ax.set_ylabel('Startup time (s)')
ax.set_title('All requests with top-1% outliers highlighted')
ax.legend(fontsize=9, markerscale=4)
plt.tight_layout()
plt.show()